# Module 4: Open Quantum Systems, Dissipation & Trajectories
## Notebook 4.4: The Monte Carlo Wavefunction (Quantum Jump) Method

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_4_open_systems_dissipation/05_monte_carlo_wavefunction_method.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** **Open Quantum Systems, Dissipation & Trajectories**
  * **4.1** Density Matrix, Purity & Entanglement Entropy
  * **4.2** The Lindblad Master Equation & Dynamical Semigroups
  * **4.3** Cavity Decay, Photon Loss & Thermal Reservoirs
  * **4.4 The Monte Carlo Wavefunction (Quantum Jump) Method** $\leftarrow$ *(You are here)*
  * **4.5** Quantum Trajectories & Homodyne Detection Unraveling
  * **4.6** Input-Output Theory & Cavity Scattering
  * **4.7** Laser Theory, Micromaser & Trapping States

---

### Learning Objectives
1. **The Curse of Dimensionality in Open Systems:** Understand why solving master equations on density matrices of dimension $d$ scales as $\mathcal{O}(d^4)$ or $\mathcal{O}(d^3)$, and why stochastic state vector unravelings ($\mathcal{O}(d^2)$) provide an exponential computational advantage.
2. **The Effective Non-Hermitian Hamiltonian:** Derive $\hat{H}_{\text{eff}} = \hat{H}_{\text{sys}} - \frac{i\hbar}{2}\sum_k \hat{L}_k^\dagger \hat{L}_k$ and interpret deterministic norm loss as the Bayesian conditioning of *no-jump* null detection.
3. **The Quantum Jump Algorithm:** Implement the Dalibard-Castin-Mølmer (MCWF) algorithm combining smooth non-unitary evolution, stochastic jump detection, and projective state collapse.
4. **Master Equation Equivalence:** Prove analytically and verify numerically that the statistical ensemble average over pure trajectories recovers the GKSL master equation: $\hat{\rho}(t) = \lim_{N_{\text{traj}}\to\infty} \frac{1}{N_{\text{traj}}}\sum_{j=1}^{N_{\text{traj}}} |\psi^{(j)}(t)\rangle\langle\psi^{(j)}(t)|$.
5. **Photon Waiting Time Distributions:** Compute the inter-jump waiting time probability distribution $w(\tau)$ and observe the signature of quantum optical antibunching ($w(0) = 0$).



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: Unraveling the Master Equation

### 1. Motivation: The Hilbert Space Bottleneck
The Gorini-Kossakowski-Sudarshan-Lindblad (GKSL) master equation governs the open-system density matrix $\hat{\rho} \in \mathbb{C}^{d \times d}$:
$$\frac{d\hat{\rho}}{dt} = -\frac{i}{\hbar}[\hat{H}, \hat{\rho}] + \sum_k \left( \hat{L}_k \hat{\rho} \hat{L}_k^\dagger - \frac{1}{2}\{\hat{L}_k^\dagger \hat{L}_k, \hat{\rho}\} \right)$$

For a composite Hilbert space of dimension $d = \prod_i d_i$, the density matrix has $d^2$ complex elements. Simulating the superoperator requires propagating a Liouvillian matrix of dimension $d^2 \times d^2$, requiring:
* **Memory Scaling:** $\mathcal{O}(d^2)$ complex numbers for the state, $\mathcal{O}(d^4)$ for full Liouvillian storage.
* **CPU Time Scaling:** $\mathcal{O}(d^3)$ to $\mathcal{O}(d^4)$ operations per time step.

For a modest multi-atom cavity system where $d = 1000$, storing $\hat{\rho}$ requires $16\text{ MB}$, but storing the Liouvillian requires $16\text{ TB}$! 
The **Monte Carlo Wavefunction (MCWF)** method, pioneered by Dalibard, Castin, and Mølmer (1992) and Carmichael (1993), bypasses this bottleneck entirely by simulating stochastic ensembles of pure state vectors $|\psi\rangle \in \mathbb{C}^d$, reducing storage to $\mathcal{O}(d)$ and matrix-vector multiplications to $\mathcal{O}(d^2)$.

---

### 2. The Effective Non-Hermitian Hamiltonian
We rewrite the dissipative anti-commutator term $-\frac{1}{2}\{\hat{L}_k^\dagger \hat{L}_k, \hat{\rho}\}$ into the coherent commutator. Defining the **effective non-Hermitian Hamiltonian**:
$$\hat{H}_{\text{eff}} \equiv \hat{H} - \frac{i\hbar}{2}\sum_k \hat{L}_k^\dagger \hat{L}_k$$

The GKSL master equation can be re-expressed as:
$$\frac{d\hat{\rho}}{dt} = -\frac{i}{\hbar}\left(\hat{H}_{\text{eff}}\hat{\rho} - \hat{\rho}\hat{H}_{\text{eff}}^\dagger\right) + \sum_k \hat{L}_k \hat{\rho} \hat{L}_k^\dagger$$

Notice the structure:
1. The first term is a non-unitary Schrödinger-like propagation driven by $\hat{H}_{\text{eff}}$.
2. The second term is a positive "recycling" source term where jump operators $\hat{L}_k$ feed dissipated population back into lower states.

---

### 3. The Dalibard-Castin-Mølmer (MCWF) Algorithm
Let the system be in a normalized pure state $|\psi(t)\rangle$ at time $t$ ($\langle\psi(t)|\psi(t)\rangle = 1$). Over a sufficiently small infinitesimal time interval $\delta t$:

#### Step A: Smooth Non-Unitary Propagation
Propagate $|\psi(t)\rangle$ forward under $\hat{H}_{\text{eff}}$:
$$|\phi(t + \delta t)\rangle = \exp\left(-\frac{i}{\hbar}\hat{H}_{\text{eff}}\delta t\right)|\psi(t)\rangle \approx \left(\hat{I} - \frac{i\delta t}{\hbar}\hat{H}_{\text{eff}}\right)|\psi(t)\rangle$$

Because $\hat{H}_{\text{eff}}$ is non-Hermitian ($\hat{H}_{\text{eff}}^\dagger \ne \hat{H}_{\text{eff}}$), the norm of $|\phi(t + \delta t)\rangle$ strictly decreases:
$$\langle\phi(t + \delta t)|\phi(t + \delta t)\rangle = 1 - \delta p + \mathcal{O}(\delta t^2)$$
where the total jump probability $\delta p$ is the sum over all jump channels $k$:
$$\delta p = \sum_k \delta p_k, \quad \delta p_k = \delta t \, \langle\psi(t)|\hat{L}_k^\dagger \hat{L}_k|\psi(t)\rangle \ge 0$$
We choose $\delta t$ sufficiently small such that $\delta p \ll 1$.

#### Step B: The Stochastic Decision (Quantum Jump vs. Null Evolution)
Draw a uniform random number $r \in [0, 1)$:

* **Case 1: Quantum Jump Occurs ($r < \delta p$)**
  A discrete quantum jump is detected by the environment! The state undergoes instantaneous projective collapse:
  1. Select the specific jump channel $m$ according to the discrete probability distribution:
     $$\Pi_m = \frac{\delta p_m}{\delta p} = \frac{\langle\psi(t)|\hat{L}_m^\dagger \hat{L}_m|\psi(t)\rangle}{\sum_k \langle\psi(t)|\hat{L}_k^\dagger \hat{L}_k|\psi(t)\rangle}$$
  2. Collapse and renormalize the state vector:
     $$|\psi(t + \delta t)\rangle = \frac{\hat{L}_m |\psi(t)\rangle}{\|\hat{L}_m |\psi(t)\rangle\|} = \frac{\hat{L}_m |\psi(t)\rangle}{\sqrt{\delta p_m / \delta t}}$$

* **Case 2: No Jump Occurs ($r \ge \delta p$)**
  No photon or emission was registered by the detector during $[t, t + \delta t]$. This is **not** an absence of measurement; it is a **null measurement**! We condition our Bayesian knowledge of the state on the fact that no jump occurred by renormalizing $|\phi(t + \delta t)\rangle$:
  $$|\psi(t + \delta t)\rangle = \frac{|\phi(t + \delta t)\rangle}{\|\phi(t + \delta t)\|} = \frac{|\phi(t + \delta t)\rangle}{\sqrt{1 - \delta p}}$$

---

### 4. Mathematical Proof of Master Equation Equivalence
Let $\overline{\hat{\sigma}(t + \delta t)}$ be the statistical expectation value of the projector $|\psi(t + \delta t)\rangle\langle\psi(t + \delta t)|$ conditioned on state $|\psi(t)\rangle$:
$$\overline{\hat{\sigma}(t + \delta t)} = (1 - \delta p) \left[ \frac{|\phi(t + \delta t)\rangle\langle\phi(t + \delta t)|}{1 - \delta p} \right] + \sum_k \delta p_k \left[ \frac{\hat{L}_k |\psi(t)\rangle\langle\psi(t)|\hat{L}_k^\dagger}{\delta p_k / \delta t} \delta t \right]$$

Canceling the scalar denominators:
$$\overline{\hat{\sigma}(t + \delta t)} = |\phi(t + \delta t)\rangle\langle\phi(t + \delta t)| + \delta t \sum_k \hat{L}_k |\psi(t)\rangle\langle\psi(t)|\hat{L}_k^\dagger$$

Expanding $|\phi(t + \delta t)\rangle$ to first order in $\delta t$:
$$|\phi(t + \delta t)\rangle = \left(\hat{I} - \frac{i\delta t}{\hbar}\hat{H} - \frac{\delta t}{2}\sum_k \hat{L}_k^\dagger \hat{L}_k\right)|\psi(t)\rangle$$
$$|\phi(t + \delta t)\rangle\langle\phi(t + \delta t)| = |\psi\rangle\langle\psi| - \frac{i\delta t}{\hbar}[\hat{H}, |\psi\rangle\langle\psi|] - \frac{\delta t}{2}\sum_k \{\hat{L}_k^\dagger \hat{L}_k, |\psi\rangle\langle\psi|\} + \mathcal{O}(\delta t^2)$$

Substituting this back into $\overline{\hat{\sigma}(t + \delta t)}$:
$$\frac{\overline{\hat{\sigma}(t + \delta t)} - |\psi(t)\rangle\langle\psi(t)|}{\delta t} = -\frac{i}{\hbar}[\hat{H}, |\psi\rangle\langle\psi|] + \sum_k \left( \hat{L}_k |\psi\rangle\langle\psi| \hat{L}_k^\dagger - \frac{1}{2}\{\hat{L}_k^\dagger \hat{L}_k, |\psi\rangle\langle\psi|\} \right)$$

Taking the ensemble average over $N_{\text{traj}}$ trajectories where $\hat{\rho}(t) = \frac{1}{N_{\text{traj}}}\sum_{j=1}^{N_{\text{traj}}} |\psi^{(j)}(t)\rangle\langle\psi^{(j)}(t)|$, we recover the **exact GKSL master equation** as $N_{\text{traj}} \to \infty$ and $\delta t \to 0$. By the Central Limit Theorem, the statistical uncertainty scales as $\sigma / \sqrt{N_{\text{traj}}}$.



In [ ]:
# Implementation: High-Performance MCWF (Quantum Jump) Engine for Driven Two-Level Atoms

class MCWFSimulator:
    """
    Monte Carlo Wavefunction (Quantum Jump) Engine for driven-dissipative open quantum systems.
    
    Default Model: Driven two-level atom with Rabi frequency Omega, detuning Delta,
    and spontaneous emission rate Gamma:
        H_sys = (Delta/2)*sigma_z + (Omega/2)*sigma_x
        L_1 = sqrt(Gamma) * sigma_- = sqrt(Gamma) * |g><e|
    """
    def __init__(self, omega=2.0 * np.pi * 1.0, delta=0.0, gamma=1.0):
        self.omega = float(omega)
        self.delta = float(delta)
        self.gamma = float(gamma)
        
        # Basis: |e> = [1, 0]^T, |g> = [0, 1]^T
        self.dim = 2
        self.sigma_z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
        self.sigma_x = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
        self.sigma_minus = np.array([[0.0, 0.0], [1.0, 0.0]], dtype=complex)  # |g><e|
        self.sigma_plus = np.array([[0.0, 1.0], [0.0, 0.0]], dtype=complex)   # |e><g|
        
        # Hamiltonian & Jump Operator
        self.H_sys = 0.5 * self.delta * self.sigma_z + 0.5 * self.omega * self.sigma_x
        self.L_ops = [np.sqrt(self.gamma) * self.sigma_minus]
        
        # Effective non-Hermitian Hamiltonian: H_eff = H_sys - (i/2) * sum_k L_k^dagger L_k
        L_dag_L_sum = sum(np.dot(L.conj().T, L) for L in self.L_ops)
        self.H_eff = self.H_sys - 0.5j * L_dag_L_sum

    def run_trajectory(self, psi0, t_list, seed=None):
        """
        Simulate a single Monte Carlo quantum jump trajectory.
        
        Parameters:
            psi0: Initial normalized state vector (shape: (2,))
            t_list: 1D array of evaluation time points (must be equally spaced)
            seed: Random seed for reproducibility
            
        Returns:
            dict containing:
                't': time array
                'P_e': excited state population trajectory |<e|psi(t)>|^2
                'jump_times': list of time stamps where quantum jumps occurred
                'jump_channels': list of channel indices that fired
        """
        rng = np.random.default_rng(seed)
        n_steps = len(t_list)
        dt = t_list[1] - t_list[0]
        
        # Precompute short-time non-unitary propagator U_eff(dt) = exp(-i * H_eff * dt)
        # Using exact matrix exponential guarantees numerical stability
        U_eff = la.expm(-1j * self.H_eff * dt)
        
        psi = np.array(psi0, dtype=complex) / la.norm(psi0)
        P_e = np.zeros(n_steps, dtype=float)
        jump_times = []
        jump_channels = []
        
        P_e[0] = np.abs(psi[0])**2
        
        for idx in range(1, n_steps):
            t_curr = t_list[idx]
            # Step A: Propagate under H_eff
            phi = np.dot(U_eff, psi)
            norm_sq = float(np.real(np.dot(phi.conj().T, phi)))
            delta_p = 1.0 - norm_sq  # Total jump probability
            
            # Step B: Stochastic decision
            r = rng.uniform(0.0, 1.0)
            if r < delta_p and delta_p > 1e-15:
                # Quantum jump occurs!
                jump_times.append(t_curr)
                # Compute individual channel probabilities
                channel_weights = []
                collapsed_states = []
                for L in self.L_ops:
                    L_phi = np.dot(L, psi)
                    weight = float(np.real(np.dot(L_phi.conj().T, L_phi))) * dt
                    channel_weights.append(weight)
                    collapsed_states.append(L_phi)
                
                tot_weight = sum(channel_weights)
                if tot_weight > 1e-16:
                    probs = np.array(channel_weights) / tot_weight
                else:
                    probs = np.ones(len(self.L_ops)) / len(self.L_ops)
                
                ch = rng.choice(len(self.L_ops), p=probs)
                jump_channels.append(ch)
                # Project and renormalize state
                psi = collapsed_states[ch] / la.norm(collapsed_states[ch])
            else:
                # No jump occurred (null measurement Bayesian update)
                psi = phi / np.sqrt(norm_sq)
            
            P_e[idx] = np.abs(psi[0])**2
            
        return {
            "t": t_list,
            "P_e": P_e,
            "jump_times": np.array(jump_times),
            "jump_channels": np.array(jump_channels)
        }

    def run_ensemble(self, psi0, t_list, n_trajectories=100, base_seed=42):
        """
        Run an ensemble of MCWF trajectories and compute statistical averages.
        """
        n_steps = len(t_list)
        P_e_matrix = np.zeros((n_trajectories, n_steps), dtype=float)
        all_jump_intervals = []
        
        for k in range(n_trajectories):
            seed = base_seed + k if base_seed is not None else None
            res = self.run_trajectory(psi0, t_list, seed=seed)
            P_e_matrix[k, :] = res["P_e"]
            
            # Record waiting times between consecutive jumps
            jt = res["jump_times"]
            if len(jt) > 1:
                all_jump_intervals.extend(np.diff(jt))
                
        mean_P_e = np.mean(P_e_matrix, axis=0)
        sem_P_e = np.std(P_e_matrix, axis=0) / np.sqrt(n_trajectories)
        
        return {
            "t": t_list,
            "mean_P_e": mean_P_e,
            "sem_P_e": sem_P_e,
            "trajectories": P_e_matrix,
            "waiting_times": np.array(all_jump_intervals)
        }

    def solve_lindblad_exact(self, psi0, t_list):
        """
        Solve the exact GKSL master equation using vectorized Liouvillian.
        """
        rho0 = np.outer(psi0, psi0.conj())
        d = self.dim
        Id = np.eye(d, dtype=complex)
        
        # Build Liouvillian superoperator: L = -i*(H (x) I - I (x) H^T) + sum_k [conj(L_k) (x) L_k - 0.5*(I (x) L_k^dag L_k + L_k^T conj(L_k) (x) I)]
        L_super = -1j * (np.kron(self.H_sys, Id) - np.kron(Id, self.H_sys.T))
        for L in self.L_ops:
            L_dag = L.conj().T
            L_super += np.kron(L.conj(), L)
            L_super -= 0.5 * np.kron(Id, np.dot(L_dag, L).T)
            L_super -= 0.5 * np.kron(np.dot(L_dag, L).conj(), Id)
            
        # Propagate using matrix exponential for high precision
        rho_vec0 = rho0.reshape(d * d, order='F')
        P_e_exact = np.zeros(len(t_list), dtype=float)
        
        for idx, t in enumerate(t_list):
            exp_Lt = la.expm(L_super * t)
            rho_vec_t = np.dot(exp_Lt, rho_vec0)
            rho_t = rho_vec_t.reshape((d, d), order='F')
            P_e_exact[idx] = np.real(rho_t[0, 0])
            
        return P_e_exact

    def analytical_steady_state_Pe(self):
        """
        Analytical steady-state excited population from optical Bloch equations:
        rho_ee^ss = (Omega^2 / 4) / (Delta^2 + Gamma^2/4 + Omega^2 / 2)
        """
        num = 0.25 * self.omega**2
        denom = self.delta**2 + 0.25 * self.gamma**2 + 0.5 * self.omega**2
        return num / denom

print("MCWFSimulator engine successfully compiled and loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: MCWF Trajectories, Ensemble Convergence & Waiting Times

# Physical parameters: Strong driving regime Omega = 2.5 * Gamma, on resonance Delta = 0
omega_val = 2.5
gamma_val = 1.0
delta_val = 0.0
sim = MCWFSimulator(omega=omega_val, delta=delta_val, gamma=gamma_val)

# Simulation time grid
t_span = np.linspace(0.0, 10.0, 1001)
dt_val = t_span[1] - t_span[0]
psi_ground = np.array([0.0, 1.0], dtype=complex)  # Start in |g>

# 1. Exact Lindblad benchmark
P_e_exact = sim.solve_lindblad_exact(psi_ground, t_span)
P_e_ss_analytic = sim.analytical_steady_state_Pe()

# 2. Compute individual trajectories
traj_1 = sim.run_trajectory(psi_ground, t_span, seed=101)
traj_2 = sim.run_trajectory(psi_ground, t_span, seed=202)
traj_3 = sim.run_trajectory(psi_ground, t_span, seed=303)

# 3. Compute ensembles for various N_traj
ens_10 = sim.run_ensemble(psi_ground, t_span, n_trajectories=10, base_seed=1000)
ens_50 = sim.run_ensemble(psi_ground, t_span, n_trajectories=50, base_seed=2000)
ens_500 = sim.run_ensemble(psi_ground, t_span, n_trajectories=500, base_seed=3000)

# 4. Long run for photon waiting time statistics
long_t_span = np.linspace(0.0, 200.0, 20001)
long_ens = sim.run_ensemble(psi_ground, long_t_span, n_trajectories=30, base_seed=777)
waiting_times = long_ens["waiting_times"]

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): Individual Quantum Jump Trajectories
# -------------------------------------------------------------
ax_a = axes[0, 0]
ax_a.plot(t_span, traj_1["P_e"], color="#e74c3c", alpha=0.85, label="Trajectory 1")
ax_a.plot(t_span, traj_2["P_e"], color="#3498db", alpha=0.75, label="Trajectory 2")
ax_a.plot(t_span, traj_3["P_e"], color="#2ecc71", alpha=0.65, label="Trajectory 3")
ax_a.plot(t_span, P_e_exact, color="black", linestyle="--", linewidth=2.4, label="Lindblad Ensemble Average")

# Mark quantum jumps for trajectory 1
for jt in traj_1["jump_times"]:
    if jt <= 10.0:
        ax_a.axvline(jt, color="#e74c3c", linestyle=":", alpha=0.5, linewidth=1.2)
        ax_a.scatter([jt], [0.0], color="#e74c3c", marker="v", s=40, zorder=5)

ax_a.set_title(r"(a) Single Stochastic Trajectories vs. Ensemble Average", fontweight="bold")
ax_a.set_xlabel(r"Normalized Time $\Gamma t$")
ax_a.set_ylabel(r"Excited State Population $P_e(t)$")
ax_a.set_ylim(-0.05, 1.05)
ax_a.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Ensemble Average Convergence with N_traj
# -------------------------------------------------------------
ax_b = axes[0, 1]
ax_b.plot(t_span, P_e_exact, color="black", linewidth=2.5, label="Exact GKSL Solution", zorder=4)
ax_b.plot(t_span, ens_10["mean_P_e"], color="#e67e22", label=r"$N_{\rm traj} = 10$", alpha=0.85)
ax_b.plot(t_span, ens_50["mean_P_e"], color="#9b59b6", label=r"$N_{\rm traj} = 50$", alpha=0.85)
ax_b.plot(t_span, ens_500["mean_P_e"], color="#16a085", label=r"$N_{\rm traj} = 500$", alpha=0.9)
ax_b.fill_between(t_span, ens_500["mean_P_e"] - ens_500["sem_P_e"], 
                  ens_500["mean_P_e"] + ens_500["sem_P_e"], color="#16a085", alpha=0.25, label=r"$\pm 1\sigma$ Error ($N=500$)")
ax_b.axhline(P_e_ss_analytic, color="gray", linestyle=":", label=r"Analytic $P_e^{\rm ss} \approx " + f"{P_e_ss_analytic:.3f}" + r"$")

ax_b.set_title(r"(b) Convergence of MCWF Average to Master Equation", fontweight="bold")
ax_b.set_xlabel(r"Normalized Time $\Gamma t$")
ax_b.set_ylabel(r"Ensemble Population $\langle P_e(t) \rangle$")
ax_b.set_ylim(-0.02, 0.85)
ax_b.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (c): Statistical Convergence Scaling (Central Limit Theorem)
# -------------------------------------------------------------
ax_c = axes[1, 0]
traj_counts = [5, 10, 25, 50, 100, 200, 500, 1000]
rmse_list = []

for N_c in traj_counts:
    ens_c = sim.run_ensemble(psi_ground, t_span, n_trajectories=N_c, base_seed=5000 + N_c)
    rmse = np.sqrt(np.mean((ens_c["mean_P_e"] - P_e_exact)**2))
    rmse_list.append(rmse)

ax_c.loglog(traj_counts, rmse_list, "o-", color="#2980b9", linewidth=2.0, markersize=7, label=r"MCWF Empirical RMSE")

# Fit power law line N^(-1/2)
fit_prefactor = rmse_list[0] * np.sqrt(traj_counts[0])
theoretical_scaling = fit_prefactor / np.sqrt(traj_counts)
ax_c.loglog(traj_counts, theoretical_scaling, "--", color="#e74c3c", linewidth=2.0, label=r"CLT Theory $\propto 1/\sqrt{N_{\rm traj}}$")

ax_c.set_title(r"(c) Statistical Error Scaling vs. $N_{\rm traj}$", fontweight="bold")
ax_c.set_xlabel(r"Number of Trajectories $N_{\rm traj}$")
ax_c.set_ylabel(r"Root-Mean-Square Error (RMSE)")
ax_c.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Photon Inter-Jump Waiting Time Distribution w(tau)
# -------------------------------------------------------------
ax_d = axes[1, 1]
# Analytical waiting time distribution for two-level atom starting from |g>:
# w(tau) = Gamma * |<e| exp(-i * H_eff * tau) |g>|^2
tau_eval = np.linspace(0.0, 6.0, 300)
w_analytic = np.zeros(len(tau_eval), dtype=float)
psi_g = np.array([0.0, 1.0], dtype=complex)

for idx, tau in enumerate(tau_eval):
    prop = la.expm(-1j * sim.H_eff * tau)
    state_tau = np.dot(prop, psi_g)
    w_analytic[idx] = sim.gamma * np.abs(state_tau[0])**2  # Emission occurs from |e>

# Plot histogram of sampled waiting times
if len(waiting_times) > 50:
    counts, bins, _ = ax_d.hist(waiting_times, bins=40, range=(0, 6), density=True, 
                               color="#34495e", alpha=0.65, edgecolor="white", label="MCWF Sampled Jumps")

ax_d.plot(tau_eval, w_analytic, color="#e74c3c", linewidth=2.4, label=r"Analytic $w(\tau) = \Gamma |\langle e| e^{-i\hat{H}_{\rm eff}\tau}|g\rangle|^2$")
ax_d.annotate("Quantum Antibunching:\n$w(0) = 0$ (Zero Simultaneous Photons)", 
              xy=(0.0, 0.0), xytext=(1.2, 0.4),
              arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#fff9c4", ec="#fbc02d", lw=1.2),
              fontweight="bold")

ax_d.set_title(r"(d) Photon Waiting Time Distribution $w(\tau)$", fontweight="bold")
ax_d.set_xlabel(r"Inter-Jump Delay $\Gamma \tau$")
ax_d.set_ylabel(r"Probability Density $w(\tau)$")
ax_d.set_xlim(-0.1, 6.0)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: Why Does the Wavefunction Jump? The Physics of Continuous Null-Measurement and Environment Unraveling**
>
> 1. **Measurement-Induced Evolution Without Photons:** A common misconception is that the environment only affects the system when an energy exchange occurs (e.g., when a detector clicks). In reality, the absence of a click is equally informative! When an atom is continuously monitored by photodetectors and no photon is detected during time $[t, t + \delta t]$, our subjective knowledge updates: the atom is *less likely* to be in the excited state $|e\rangle$. This Bayesian inference is mathematically represented by the non-Hermitian operator $\hat{H}_{\text{eff}} = \hat{H} - \frac{i}{2}\Gamma |e\rangle\langle e|$, which progressively damps the excited-state amplitude even when no jump occurs.
> 2. **Photon Antibunching and the Reset Condition:** Immediately following a quantum jump $\hat{L} = \sqrt{\Gamma}|g\rangle\langle e|$, the atom collapses strictly into the ground state $|g\rangle$. Because a second photon can only be emitted once the atom is re-excited to $|e\rangle$ by the coherent Rabi drive, the probability density of an immediate second jump is identically zero: $w(0) = 0$. This manifests the celebrated phenomenon of **resonance fluorescence photon antibunching** ($g^{(2)}(0) = 0$), discovered by Kimble, Dagenais, and Mandel in 1977.
> 3. **The Wavefunction as a State of Knowledge:** A single MCWF trajectory represents the conditional state of an individual quantum system conditioned on a specific experimental realization of detector counts. The density matrix $\hat{\rho}(t)$, by contrast, represents the non-selective ensemble average over all possible measurement records.



In [ ]:
# Interactive Exploration: Stochastic Quantum Jumps & Driven Dynamics

def interactive_mcwf_explorer(omega_over_gamma=2.5, delta_over_gamma=0.0, n_trajectories=15, seed=42):
    """
    Interactive exploration of single trajectories and ensemble convergence.
    """
    gamma = 1.0
    omega = omega_over_gamma * gamma
    delta = delta_over_gamma * gamma
    
    sim = MCWFSimulator(omega=omega, delta=delta, gamma=gamma)
    t_span = np.linspace(0.0, 10.0, 501)
    psi_g = np.array([0.0, 1.0], dtype=complex)
    
    # Run ensemble
    ens = sim.run_ensemble(psi_g, t_span, n_trajectories=n_trajectories, base_seed=seed)
    P_e_exact = sim.solve_lindblad_exact(psi_g, t_span)
    P_e_ss = sim.analytical_steady_state_Pe()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: All individual trajectories
    for k in range(min(n_trajectories, 10)):
        ax1.plot(t_span, ens["trajectories"][k], alpha=0.5, linewidth=1.2)
    ax1.plot(t_span, P_e_exact, "k--", linewidth=2.5, label="Master Eq.")
    ax1.set_title(f"First {min(n_trajectories, 10)} Individual Trajectories")
    ax1.set_xlabel(r"$\Gamma t$")
    ax1.set_ylabel(r"$P_e(t)$")
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(loc="upper right")
    
    # Subplot 2: Ensemble average vs exact
    ax2.plot(t_span, P_e_exact, "k-", linewidth=2.5, label="Master Eq. (Exact)")
    ax2.plot(t_span, ens["mean_P_e"], color="#e74c3c", linewidth=2.0, label=f"MCWF Average (N={n_trajectories})")
    ax2.fill_between(t_span, ens["mean_P_e"] - ens["sem_P_e"], 
                     ens["mean_P_e"] + ens["sem_P_e"], color="#e74c3c", alpha=0.25, label=r"$\pm 1\sigma$ SEM")
    ax2.axhline(P_e_ss, color="gray", linestyle=":", label=f"Steady-State = {P_e_ss:.3f}")
    
    ax2.set_title(r"Ensemble Average Convergence")
    ax2.set_xlabel(r"$\Gamma t$")
    ax2.set_ylabel(r"$\langle P_e(t) \rangle$")
    ax2.set_ylim(-0.05, 0.85)
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_mcwf_explorer(omega_over_gamma=2.5, delta_over_gamma=0.0, n_trajectories=20, seed=123)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_mcwf_explorer,
        omega_over_gamma=widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.5, description=r"$\Omega/\Gamma$"),
        delta_over_gamma=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description=r"$\Delta/\Gamma$"),
        n_trajectories=widgets.IntSlider(value=25, min=5, max=100, step=5, description=r"$N_{\rm traj}$"),
        seed=widgets.IntSlider(value=42, min=1, max=500, step=1, description="Seed")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: Central Limit Theorem & Error Scaling
1. Demonstrate that for any fixed evaluation time $t^*$, the sample mean $\bar{P}_e(t^*)$ computed across $N_{\text{traj}}$ trajectories satisfies the Central Limit Theorem:
   $$\text{Var}[\bar{P}_e(t^*)] = \frac{\sigma^2(t^*)}{N_{\text{traj}}} \implies \text{SEM} = \frac{\sigma(t^*)}{\sqrt{N_{\text{traj}}}}$$
2. Benchmark empirical error distributions against normal distributions $\mathcal{N}(\mu, \sigma^2/N)$.

---

### Challenge 2: Steady-State Precision Benchmark
1. In the driven two-level system under continuous resonant driving ($\Delta = 0$), show that the analytical steady-state excited population is:
   $$P_e^{(\text{ss})} = \frac{\Omega^2 / 4}{\Gamma^2 / 4 + \Omega^2 / 2} = \frac{s_0 / 2}{1 + s_0}, \quad s_0 \equiv \frac{2\Omega^2}{\Gamma^2}$$
2. For $\Omega = 2\Gamma$, compute the exact steady state $P_e^{(\text{ss})} = \frac{1}{1 + 1/4} = \frac{4}{9} \approx 0.4444$.
3. Run $N_{\text{traj}} = 500$ trajectories to long times ($\Gamma t = 15$) and verify that the late-time average agrees with the exact value within $3\sigma$ statistical confidence:
   $$\left|\langle P_e(\Gamma t \gg 1)\rangle_{\text{MCWF}} - P_e^{(\text{ss})}\right| \leq 3 \times \text{SEM}$$

---

### Challenge 3: Continuous Monitoring & The Quantum Zeno Effect
1. Consider the strongly damped limit $\Gamma \gg \Omega$. In this regime, the continuous environment monitoring rate overwhelms the coherent Rabi driving rate.
2. Analyze the effective jump rate: show that the excitation transition rate from $|g\rangle$ to $|e\rangle$ is suppressed as $R_{\text{eff}} \approx \frac{\Omega^2}{\Gamma}$, freezing the system into the ground state $|g\rangle$ (the continuous Quantum Zeno Effect).



In [ ]:
# Numerical Verification of Challenge 2: Steady-State MCWF Precision Benchmark

omega_bench = 2.0
gamma_bench = 1.0
delta_bench = 0.0

sim_bench = MCWFSimulator(omega=omega_bench, delta=delta_bench, gamma=gamma_bench)

# Exact analytical steady-state population:
# P_e_ss = (Omega^2 / 4) / (Gamma^2/4 + Omega^2/2) = (4/4) / (1/4 + 4/2) = 1 / (0.25 + 2) = 1 / 2.25 = 4/9
P_e_ss_analytic = sim_bench.analytical_steady_state_Pe()
expected_ss = 4.0 / 9.0
assert np.isclose(P_e_ss_analytic, expected_ss, atol=1e-12), "Analytical formula mismatch!"

# Long-time simulation grid: evolve into deep steady state
t_bench = np.linspace(0.0, 15.0, 601)
n_trajectories_bench = 600

print(f"Running {n_trajectories_bench} MCWF trajectories for steady-state precision benchmark...")
ens_bench = sim_bench.run_ensemble(np.array([0.0, 1.0], dtype=complex), t_bench, 
                                   n_trajectories=n_trajectories_bench, base_seed=888)

# Compute late-time steady-state average over t in [10.0, 15.0]
late_mask = t_bench >= 10.0
mcwf_steady_mean = np.mean(ens_bench["mean_P_e"][late_mask])
mcwf_sem = np.mean(ens_bench["sem_P_e"][late_mask])

diff = np.abs(mcwf_steady_mean - expected_ss)
print(f"Analytical Steady State P_e: {expected_ss:.6f}")
print(f"MCWF Empirical Steady State: {mcwf_steady_mean:.6f} +/- {mcwf_sem:.6f}")
print(f"Difference: {diff:.6f} ({diff / mcwf_sem:.2f} standard errors)")

# Rigorous statistical assertion: Must agree within 3 standard errors (99.7% confidence)
assert diff <= 3.0 * mcwf_sem, f"MCWF steady-state deviates by {diff/mcwf_sem:.2f} sigma (> 3 sigma)!"

# Verify purity and normalization along trajectory
test_traj = sim_bench.run_trajectory(np.array([0.0, 1.0], dtype=complex), t_bench[:100], seed=999)
assert np.all(test_traj["P_e"] >= -1e-12) and np.all(test_traj["P_e"] <= 1.0 + 1e-12), "Population bounded in [0, 1] violated!"

print("=" * 70)
print("SUCCESS: MCWF Quantum Jump Method Numerically Verified & Benchmarked!")
print(f"- Analytical P_e^ss: {expected_ss:.6f}")
print(f"- MCWF P_e^ss:       {mcwf_steady_mean:.6f} +/- {mcwf_sem:.6f}")
print(f"- Agreement:         {diff / mcwf_sem:.2f} sigma (PASS: <= 3 sigma)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 4.4
1. **Computational Advantage:** Simulating open quantum dynamics via stochastic state vectors scales as $\mathcal{O}(d^2)$ instead of $\mathcal{O}(d^4)$, unlocking high-dimensional systems inaccessible to density-matrix integration.
2. **Effective Hamiltonian:** Dissipation enters deterministically through the anti-Hermitian operator $-\frac{i}{2}\sum_k \hat{L}_k^\dagger \hat{L}_k$, which continuously updates the state conditioned on no jumps being detected.
3. **Equivalence to GKSL:** The ensemble average over individual quantum jump realizations recovers the exact Lindblad master equation with $\mathcal{O}(1/\sqrt{N_{\text{traj}}})$ statistical convergence.
4. **Physical Reality of Jumps:** Inter-jump waiting time distributions reveal direct physical properties of light, including resonance fluorescence photon antibunching ($w(0) = 0$).

---

### Up Next: Notebook 4.5
In [Notebook 4.5: Quantum Trajectories & Homodyne Detection Unraveling](05_quantum_trajectories_and_homodyne_measurement.ipynb), we transition from discrete quantum jumps (photo-counting) to **continuous diffusive measurement unravelings** (balanced homodyne and heterodyne detection), exploring the Stochastic Schrödinger Equation (SSE), measurement backaction, and conditional squeezing.

